# Tutorial: Clasificaci?n de abandono de empleado (Employee Churn Model)

## Contexto del ejercicio
En este notebook construiremos un modelo de clasificaci?n para estimar la probabilidad de que un empleado abandone la organizaci?n. Este tipo de ejercicio es ?til en iniciativas de *People Analytics*, retenci?n de talento y an?lisis predictivo aplicado a recursos humanos.

## Audiencia
- Alumnos de posgrado y educaci?n continua que desean practicar clasificaci?n supervisada con datos tabulares.

## Prerrequisitos
- Conocimientos b?sicos de Python y pandas.
- Nociones generales de entrenamiento y evaluaci?n de modelos de clasificaci?n.

## Objetivo
Construir un modelo base de **abandono de empleado** usando `scikit-learn`, evaluar su desempe?o e interpretar qu? variables se relacionan con la salida del personal.


## Ruta del ejercicio
1. Preparar dependencias y entorno.
2. Cargar el dataset en Google Colab o desde GitHub.
3. Explorar la estructura del problema y la variable objetivo.
4. Preparar variables num?ricas y categ?ricas.
5. Entrenar un modelo de clasificaci?n con `LogisticRegression`.
6. Evaluar el modelo con m?tricas y matriz de confusi?n.
7. Interpretar variables relevantes y discutir implicaciones de negocio.


## C?mo usar el dataset en Google Colab
Tiene dos opciones:
- **Opci?n 1:** usar autom?ticamente el archivo publicado en GitHub.
- **Opci?n 2:** subir manualmente el CSV a Colab desde su computadora.

En este notebook la opci?n predeterminada es usar GitHub, pero puede activar la carga manual cambiando `USE_COLAB_UPLOAD = True`.


In [ ]:
import sys

if 'google.colab' in sys.modules:
    try:
        import sklearn  # noqa: F401
    except ImportError:
        %pip install -q scikit-learn pandas matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

plt.style.use('seaborn-v0_8-whitegrid')


In [ ]:
RAW_DATA_URL = 'https://raw.githubusercontent.com/JuanBaldemarG/portafoliocolabJBGV/main/data/employee-churn/HR_dataset_copy.csv'
USE_COLAB_UPLOAD = False
LOCAL_CANDIDATES = [
    Path('../data/employee-churn/HR_dataset_copy.csv'),
    Path('data/employee-churn/HR_dataset_copy.csv'),
    Path('/content/HR_dataset_copy.csv')
]

def load_dataset() -> pd.DataFrame:
    if 'google.colab' in sys.modules:
        if USE_COLAB_UPLOAD:
            from google.colab import files
            uploaded = files.upload()
            uploaded_name = next(iter(uploaded))
            return pd.read_csv(uploaded_name)

        for candidate in LOCAL_CANDIDATES:
            if candidate.exists():
                return pd.read_csv(candidate)

        return pd.read_csv(RAW_DATA_URL)

    for candidate in LOCAL_CANDIDATES:
        if candidate.exists():
            return pd.read_csv(candidate)

    return pd.read_csv(RAW_DATA_URL)

df = load_dataset()
df.head()


## Exploraci?n inicial
Antes de modelar, conviene entender cu?ntos registros tenemos, qu? columnas existen y qu? tipos de datos est?n presentes. Esto ayuda a decidir c?mo preparar el pipeline.


In [ ]:
print(f'Registros: {df.shape[0]:,}')
print(f'Columnas: {df.shape[1]}')
display(df.dtypes.to_frame('tipo'))


In [ ]:
target = 'left'
target_distribution = df[target].value_counts().sort_index()
target_share = df[target].value_counts(normalize=True).sort_index()
display(pd.DataFrame({'conteo': target_distribution, 'proporci?n': target_share}))

ax = target_distribution.plot(kind='bar', color=['#4c78a8', '#f58518'], figsize=(6, 4))
ax.set_title('Distribuci?n de la variable objetivo: left')
ax.set_xlabel('Clase')
ax.set_ylabel('N?mero de empleados')
plt.show()


### Interpretaci?n inicial
La variable `left` representa si el empleado sali? de la organizaci?n (`1`) o permaneci? (`0`). Esta distribuci?n es importante porque una base muy desbalanceada puede afectar la interpretaci?n de m?tricas como *accuracy*.


In [ ]:
summary_cols = ['satisfaction_level', 'last_evaluation', 'number_project', 'average_montly_hours', 'time_spend_company']
df.groupby(target)[summary_cols].mean().round(2)


## Preparaci?n del modelo
Usaremos la variable `left` como objetivo. Las columnas num?ricas y categ?ricas se transforman dentro de un `Pipeline` para dejar el flujo reproducible y compatible con Colab.


In [ ]:
X = df.drop(columns=[target])
y = df[target]

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'Tama?o entrenamiento: {X_train.shape[0]:,}')
print(f'Tama?o prueba: {X_test.shape[0]:,}')
print(f'Variables num?ricas: {len(numeric_features)}')
print(f'Variables categ?ricas: {len(categorical_features)}')


In [ ]:
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred)
roc_auc = roc_auc_score(y_test, proba)

print(f'Accuracy: {accuracy:.4f}')
print(f'ROC AUC: {roc_auc:.4f}')
print()
print(classification_report(y_test, pred))


### C?mo interpretar estas m?tricas
- **Accuracy:** proporci?n total de predicciones correctas.
- **Precision:** de los empleados marcados como abandono, cu?ntos realmente abandonaron.
- **Recall:** de los empleados que realmente abandonaron, cu?ntos fueron detectados por el modelo.
- **ROC AUC:** capacidad general del modelo para distinguir entre permanencia y abandono a distintos umbrales.

En problemas de retenci?n suele ser especialmente importante vigilar el **recall** de la clase de abandono, porque un falso negativo significa no detectar a tiempo a un empleado con riesgo de salida.


In [ ]:
cm = confusion_matrix(y_test, pred)
cm_df = pd.DataFrame(cm, index=['Real 0', 'Real 1'], columns=['Pred 0', 'Pred 1'])
display(cm_df)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusi?n')
plt.show()


### Interpretaci?n de la matriz de confusi?n
- **Pred 1 / Real 1:** empleados con abandono correctamente detectados.
- **Pred 0 / Real 1:** empleados que abandonaron pero el modelo no detect?.
- **Pred 1 / Real 0:** empleados marcados con riesgo aunque en realidad permanecieron.

Desde una perspectiva de negocio, los falsos negativos suelen ser m?s costosos si la organizaci?n quiere intervenir antes de perder talento clave.


In [ ]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefficients = model.named_steps['classifier'].coef_[0]
coef_df = (
    pd.DataFrame({'feature': feature_names, 'coef': coefficients})
    .assign(abs_coef=lambda d: d['coef'].abs())
    .sort_values('abs_coef', ascending=False)
)
coef_df[['feature', 'coef']].head(12)


## Interpretaci?n de variables relevantes
En una regresi?n log?stica, un coeficiente positivo empuja la predicci?n hacia `left = 1` y un coeficiente negativo hacia `left = 0`.

Esto no debe leerse como causalidad directa, sino como una se?al estad?stica dentro de este dataset. Conviene complementar estos hallazgos con conocimiento del proceso, entrevistas y pol?ticas de recursos humanos.


In [ ]:
top_positive = coef_df.sort_values('coef', ascending=False).head(5)[['feature', 'coef']]
top_negative = coef_df.sort_values('coef', ascending=True).head(5)[['feature', 'coef']]

print('Variables m?s asociadas con abandono (coeficientes positivos):')
display(top_positive)

print('Variables m?s asociadas con permanencia (coeficientes negativos):')
display(top_negative)


## Conclusi?n ejecutiva
Este notebook deja una l?nea base reproducible para el problema de abandono de empleado. A partir de los resultados obtenidos, el grupo puede discutir preguntas como:
- ?Qu? tan ?til es el modelo para priorizar intervenciones de retenci?n?
- ?Conviene optimizar el modelo hacia mayor recall o mayor precisi?n?
- ?Qu? variables merecen revisi?n por parte del ?rea de recursos humanos?

El siguiente paso natural ser?a comparar este modelo con alternativas como ?rboles, bosques aleatorios o *gradient boosting*.


## Ejercicio para el alumno
Pruebe una de estas extensiones:
1. Cambiar `LogisticRegression` por `RandomForestClassifier`.
2. Ajustar el umbral de clasificaci?n usando `predict_proba`.
3. Comparar resultados quitando la columna categ?rica `functional area`.
4. Evaluar si la satisfacci?n del empleado parece ser una variable especialmente sensible.


In [ ]:
# Respuesta sugerida: use este espacio para probar un segundo modelo o un nuevo umbral.
# Ejemplo:
# from sklearn.ensemble import RandomForestClassifier
# ...


## Errores comunes y extensiones
**Error com?n:** olvidar el tratamiento de variables categ?ricas y pasar texto crudo al modelo.

**Error com?n:** quedarse solo con accuracy y no revisar recall, precisi?n o matriz de confusi?n.

**Extensi?n sugerida:** agregar validaci?n cruzada, comparar varios clasificadores y documentar cu?l conviene presentar como modelo final seg?n el objetivo del negocio.
